# 3. Korištenje modela

Model je naučen i testiran – vrijeme je da ga upotrijebimo! Za svaku sliku model izračuna **vjerojatnost** za svaki od 10 razreda, a mi prikazujemo tri najvjerojatnija.

Ako dobiješ grešku `ModuleNotFoundError`, najprije pokreni 1. bilježnicu (ona instalira potrebne pakete).

## Korak 0: Učitavanje modela

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image

import eurosat_common as ec

device = ec.get_device()
model = ec.load_model(ec.MODEL_PATH, device)

## Korak 1: Funkcija za prikaz predviđanja

Lijevo je slika onakva kakvu je model "vidi" (64 × 64 piksela), a desno tri najvjerojatnija razreda.

In [ ]:
def show_prediction(image, title="", true_class=None):
    """Prikaži sliku i tri najvjerojatnija razreda prema modelu."""
    predictions = ec.predict_image(model, image, device, top_k=3)
    names = [name for name, _ in predictions]
    percents = [probability * 100 for _, probability in predictions]

    fig, (ax_image, ax_bars) = plt.subplots(1, 2, figsize=(11, 3), gridspec_kw={"width_ratios": [1, 2.5]})

    ax_image.imshow(ec.tensor_to_image(ec.eval_transform(image.convert("RGB"))))
    ax_image.set_title(title, fontsize=9)
    ax_image.axis("off")

    ax_bars.barh(names, percents, color="#2a78d6")
    for i, percent in enumerate(percents):
        ax_bars.text(percent + 1, i, f"{percent:.1f} %", va="center")
    ax_bars.invert_yaxis()
    ax_bars.set_xlim(0, 112)
    ax_bars.set_xlabel("vjerojatnost (%)")
    if true_class is not None:
        verdict = "TOČNO" if names[0] == true_class else "POGREŠNO"
        ax_bars.set_title(f"Pravi razred: {true_class}  ->  {verdict}")

    plt.tight_layout()
    plt.show()

## Korak 2: Primjeri iz testnog skupa

Svaki put kad pokreneš ćeliju, dobiješ 6 novih nasumičnih slika.

In [ ]:
test_data = load_dataset(ec.DATASET_ID, split="test")

for index in random.sample(range(len(test_data)), 6):
    example = test_data[index]
    show_prediction(example["image"], title=f"testna slika br. {index}",
                    true_class=ec.CLASS_NAMES[example["label"]])

## Korak 3: Tvoje vlastite slike

1. Nađi satelitsku snimku, npr. u [Google Maps](https://maps.google.com) (prikaz *Satelit*) ili u [Copernicus Browseru](https://browser.dataspace.copernicus.eu) (prave snimke satelita Sentinel-2!).
2. Približi tako da na ekranu bude otprilike **640 × 640 m** terena – toliko pokriva jedna EuroSAT slika. Ako je preblizu ili predaleko, model će lošije prepoznavati.
3. Napravi snimku zaslona (najbolje kvadratnu) i spremi je kao `.jpg` ili `.png`.
4. U JupyterHubu u pregledniku datoteka (lijevo) otvori mapu `moje_slike` i povuci sliku u nju. (Mapu stvara ćelija ispod, ako još ne postoji.)
5. Pokreni ćeliju ispod.

Model iz slike izreže kvadrat iz sredine i smanji ga na 64 × 64 piksela – jednako kao slike na kojima je učio.

In [ ]:
image_folder = Path("moje_slike")
image_folder.mkdir(exist_ok=True)

image_paths = sorted(p for p in image_folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
if not image_paths:
    print(f"Mapa '{image_folder}' je prazna. Dodaj u nju slike (.jpg ili .png) pa ponovno pokreni ovu ćeliju.")

for path in image_paths:
    show_prediction(Image.open(path), title=path.name)

## Važno: model poznaje samo 10 razreda

Model **uvijek** odabere jedan od 10 razreda – čak i kad mu daš fotografiju mačke! Ne zna reći "ne znam". Niska vjerojatnost najboljeg razreda (npr. ispod 50 %) znak je da model nije siguran.

**Probaj:** stavi u mapu `moje_slike` fotografiju nečega što nije satelitska snimka i pogledaj što model kaže.